# Data-driven operator — Timon round 5, point 7b

The same Transolver, same mesh, same 800/200 split, same optimizer and
the **same optimizer-step budget** — trained on FEM solutions instead of
the energy functional. Only the loss differs, so the comparison isolates
the training principle rather than the architecture.

* **No new FEM solves** — the labels are already in the dataset.
* **GPU strongly preferred**; CPU works but will be slow.
* Expected to finish inside the physics-informed run's 48 minutes,
  since a data loss skips the per-step energy assembly.

The script also reports the **label-generation cost** the data-driven
model implicitly requires and the physics-informed one does not — about
5.7 h of CPU for 800 B1 × Neo-Hookean solves at Table 4a's measured rate.


In [ ]:
# =====================================================================
#  CELL — Data-driven operator, for comparison with the physics-informed one
#  (Timon round-5 point 7b; round-6 said to start with B1 x Neo-Hookean.)
#
#  CPU works but will be slow; a GPU runtime is strongly preferred.
#  Needs NO new FEM solves: the labels are already in the dataset the
#  physics-informed model trained on.
#  Saves a checkpoint and history to Drive at every evaluation, so a
#  disconnect loses at most `eval_every` steps of progress.
# =====================================================================
import os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
                    'https://github.com/SUHIBAMRO/OMAR.git', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin',
                    'claude/claude-code-question-d307wp'], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout',
                    'claude/claude-code-question-d307wp'], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard',
                    'origin/claude/claude-code-question-d307wp'], check=True)

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE - this will be slow, prefer Runtime > Change runtime type > GPU')

R = '/content/drive/MyDrive/pfem_run'
OUT = f'{R}/data_driven'
os.makedirs(OUT, exist_ok=True)

# --- the case, and the settings that MUST match the physics-informed run ---
GEOMETRY  = 'B1'
MATERIAL  = 'neo_hookean'
DATA      = f'{R}/results/datasets/B1_neo_hookean/hyperelastic_training_data_q4.npz'
OPT_STEPS = 75000    # B1 x Neo-Hookean's own step count, from Table 7
BATCH     = 8        # the protocol's batch size (Table 4)
# --------------------------------------------------------------------------

assert os.path.exists(DATA), (
    f'dataset not found: {DATA}\n'
    'Check the path -- it must be the SAME .npz the physics-informed model '
    'trained on, or the comparison is not like-for-like.')
print('dataset:', DATA, f'({os.path.getsize(DATA)/1e6:.1f} MB)')

subprocess.run([
    sys.executable, '-m', 'omar_pfem.train_data_driven',
    '--geometry', GEOMETRY, '--material', MATERIAL,
    '--path', DATA,
    '--ntrain', '800', '--ntest', '200',
    '--batch_size', str(BATCH),
    '--opt_steps', str(OPT_STEPS),
    '--loss', 'rel_l2',
    '--eval_every', '2000',
    '--out_dir', f'{OUT}/{GEOMETRY}_{MATERIAL}',
], check=True)

print(f'\nDone. Result: {OUT}/{GEOMETRY}_{MATERIAL}/'
      f'data_driven_{GEOMETRY}_{MATERIAL}.json')
print('Compare its best_val_rel_L2 against Table 5 (physics-informed, 0.0959 for')
print('this case) and note label_generation_cost_h, which the physics-informed')
print('model does not pay at all.')
